# 07: The Perceptron - The Simplest "Brain Cell"

## Where Neural Networks Begin

The **perceptron** is the fundamental building block of neural networks. It's inspired by how neurons work in the brain:

1. **Receive inputs** (signals from other neurons)
2. **Weight them** (some signals matter more)
3. **Sum them up** (aggregate the information)
4. **Decide** (fire or don't fire)

### The Web Dev Analogy

Think of a perceptron like a **form validation rule**:
- Input fields have values
- Some fields matter more (required vs optional)
- Combine them with weights
- Output: Valid (1) or Invalid (0)

## What You'll Learn
- [ ] Implement a perceptron and understand its learning rule
- [ ] Explain linear separability and why it limits single neurons
- [ ] Demonstrate that XOR cannot be solved by a single perceptron

## Connection to Previous Lessons

| What you learned | How it connects here |
|-----------------|---------------------|
| **Lesson 4**: Logistic regression (`z = wx + b → σ(z)`) | A perceptron is the same formula with a step function instead of sigmoid! |

> Why go "backward" to a simpler model? Because understanding the perceptron's **limits** motivates why we need neural networks.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(42)

print("Ready to build artificial neurons! 🧠")

## 1. The Perceptron Model

A perceptron computes:

$$y = f(\sum_{i} w_i x_i + b)$$

Where:
- $x_i$ = inputs
- $w_i$ = weights (learned parameters)
- $b$ = bias (threshold)
- $f$ = activation function

In [ ]:
# Visualize a perceptron
fig, ax = plt.subplots(figsize=(10, 6))

# Input nodes
input_positions = [(0.1, 0.8), (0.1, 0.5), (0.1, 0.2)]
input_labels = ['x₁', 'x₂', 'x₃']

for pos, label in zip(input_positions, input_labels):
    circle = plt.Circle(pos, 0.05, color='lightblue', ec='black')
    ax.add_patch(circle)
    ax.text(pos[0]-0.1, pos[1], label, fontsize=14, ha='center', va='center')

# Neuron body
neuron_pos = (0.5, 0.5)
circle = plt.Circle(neuron_pos, 0.1, color='lightgreen', ec='black', linewidth=2)
ax.add_patch(circle)
ax.text(neuron_pos[0], neuron_pos[1], 'Σ', fontsize=20, ha='center', va='center')

# Connections with weights
weights = ['w₁', 'w₂', 'w₃']
for pos, w in zip(input_positions, weights):
    ax.annotate('', xy=(0.4, 0.5), xytext=(pos[0]+0.05, pos[1]),
                arrowprops=dict(arrowstyle='->', color='gray'))
    mid_x = (pos[0] + 0.4) / 2
    mid_y = (pos[1] + 0.5) / 2
    ax.text(mid_x, mid_y + 0.05, w, fontsize=10, color='red')

# Output
ax.annotate('', xy=(0.75, 0.5), xytext=(0.6, 0.5),
            arrowprops=dict(arrowstyle='->', color='gray', lw=2))

# Activation function box
ax.text(0.75, 0.5, 'f(·)', fontsize=14, ha='center', va='center',
        bbox=dict(boxstyle='round', facecolor='lightyellow', edgecolor='black'))

# Output node
output_pos = (0.9, 0.5)
circle = plt.Circle(output_pos, 0.05, color='lightcoral', ec='black')
ax.add_patch(circle)
ax.text(output_pos[0]+0.08, output_pos[1], 'ŷ', fontsize=14, ha='center', va='center')

# Bias
ax.text(0.5, 0.35, '+b', fontsize=12, ha='center', color='blue')

ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis('off')
ax.set_title('The Perceptron: A Single Artificial Neuron', fontsize=14)
plt.tight_layout()
plt.show()

## 2. Step Activation Function

The original perceptron uses a **step function**:
- Output 1 if weighted sum ≥ 0
- Output 0 otherwise

In [ ]:
def step_function(z):
    """Binary step activation: 1 if z >= 0, else 0."""
    return (z >= 0).astype(int)

# Visualize
z = np.linspace(-5, 5, 1000)
y = step_function(z)

plt.figure(figsize=(8, 4))
plt.plot(z, y, 'b-', linewidth=2)
plt.axhline(0, color='gray', linestyle=':', alpha=0.5)
plt.axhline(1, color='gray', linestyle=':', alpha=0.5)
plt.axvline(0, color='red', linestyle='--', alpha=0.5, label='Threshold')
plt.xlabel('z (weighted sum)')
plt.ylabel('Output')
plt.title('Step Activation Function')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("If sum of weighted inputs ≥ 0 → Fire (1)")
print("If sum of weighted inputs < 0 → Don't fire (0)")

## 3. Learning Logic Gates

Let's train a perceptron to learn basic logic gates!

In [ ]:
class Perceptron:
    """A single perceptron (artificial neuron)."""
    
    def __init__(self, n_inputs, learning_rate=0.1):
        self.weights = np.zeros(n_inputs)
        self.bias = 0
        self.lr = learning_rate
    
    def predict(self, X):
        """Make predictions."""
        z = np.dot(X, self.weights) + self.bias
        return step_function(z)
    
    def train(self, X, y, epochs=100):
        """Train using the perceptron learning rule."""
        history = []
        
        for epoch in range(epochs):
            errors = 0
            
            for xi, yi in zip(X, y):
                # Make prediction
                prediction = self.predict(xi)
                
                # Calculate error
                error = yi - prediction
                
                if error != 0:
                    errors += 1
                    # Update weights and bias
                    self.weights += self.lr * error * xi
                    self.bias += self.lr * error
            
            accuracy = 1 - errors / len(y)
            history.append(accuracy)
            
            if errors == 0:
                print(f"Converged at epoch {epoch + 1}!")
                break
        
        return history

In [ ]:
# AND gate data
X_and = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y_and = np.array([0, 0, 0, 1])  # AND: both must be 1

# OR gate data
X_or = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y_or = np.array([0, 1, 1, 1])  # OR: at least one must be 1

print("AND Gate Truth Table:")
for x, y in zip(X_and, y_and):
    print(f"  {x[0]} AND {x[1]} = {y}")

print("\nOR Gate Truth Table:")
for x, y in zip(X_or, y_or):
    print(f"  {x[0]} OR {x[1]} = {y}")

In [ ]:
# Train perceptrons for AND and OR
print("Training AND perceptron:")
and_perceptron = Perceptron(2)
and_history = and_perceptron.train(X_and, y_and)

print(f"\nFinal weights: {and_perceptron.weights}, bias: {and_perceptron.bias}")
print("\nPredictions:")
for x, y in zip(X_and, y_and):
    pred = and_perceptron.predict(x)
    status = "✓" if pred == y else "✗"
    print(f"  {x} → {pred} (expected {y}) {status}")

In [ ]:
print("\nTraining OR perceptron:")
or_perceptron = Perceptron(2)
or_history = or_perceptron.train(X_or, y_or)

print(f"\nFinal weights: {or_perceptron.weights}, bias: {or_perceptron.bias}")
print("\nPredictions:")
for x, y in zip(X_or, y_or):
    pred = or_perceptron.predict(x)
    status = "✓" if pred == y else "✗"
    print(f"  {x} → {pred} (expected {y}) {status}")

## 4. Visualizing Decision Boundaries

In [ ]:
def plot_decision_boundary(perceptron, X, y, title):
    """Plot the decision boundary of a perceptron."""
    fig, ax = plt.subplots(figsize=(6, 6))
    
    # Create mesh grid
    x_min, x_max = -0.5, 1.5
    y_min, y_max = -0.5, 1.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200),
                         np.linspace(y_min, y_max, 200))
    
    # Predict on mesh
    Z = perceptron.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    
    # Plot decision regions
    ax.contourf(xx, yy, Z, levels=[-0.5, 0.5, 1.5], colors=['lightcoral', 'lightgreen'], alpha=0.6)
    ax.contour(xx, yy, Z, levels=[0.5], colors=['black'], linewidths=2)
    
    # Plot data points
    for xi, yi in zip(X, y):
        color = 'green' if yi == 1 else 'red'
        marker = 'o' if yi == 1 else 's'
        ax.scatter(xi[0], xi[1], c=color, s=200, marker=marker, edgecolors='black', zorder=3)
    
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)
    ax.set_xlabel('Input 1')
    ax.set_ylabel('Input 2')
    ax.set_title(title)
    ax.grid(True, alpha=0.3)
    
    return fig, ax

# Plot both
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# AND
plt.sca(axes[0])
plot_decision_boundary(and_perceptron, X_and, y_and, 'AND Gate')

# OR
plt.sca(axes[1])
plot_decision_boundary(or_perceptron, X_or, y_or, 'OR Gate')

plt.tight_layout()
plt.show()

print("🔑 The perceptron learned a LINEAR decision boundary!")
print("   Green = Output 1, Red = Output 0")

## 5. The XOR Problem: Perceptron's Limitation

The XOR gate cannot be learned by a single perceptron!

In [ ]:
# XOR gate data
X_xor = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y_xor = np.array([0, 1, 1, 0])  # XOR: exactly one must be 1

print("XOR Gate Truth Table:")
for x, y in zip(X_xor, y_xor):
    print(f"  {x[0]} XOR {x[1]} = {y}")

# Try to train
print("\nTraining XOR perceptron:")
xor_perceptron = Perceptron(2)
xor_history = xor_perceptron.train(X_xor, y_xor, epochs=1000)

print(f"\nFinal accuracy: {xor_history[-1]:.2%}")
print("\nPredictions:")
for x, y in zip(X_xor, y_xor):
    pred = xor_perceptron.predict(x)
    status = "✓" if pred == y else "✗"
    print(f"  {x} → {pred} (expected {y}) {status}")

In [ ]:
# Visualize why XOR can't be solved
fig, ax = plt.subplots(figsize=(7, 6))

# Plot XOR points
for xi, yi in zip(X_xor, y_xor):
    color = 'green' if yi == 1 else 'red'
    ax.scatter(xi[0], xi[1], c=color, s=300, edgecolors='black', zorder=3)
    ax.annotate(f'{xi}→{yi}', (xi[0]+0.1, xi[1]+0.1), fontsize=12)

# Try to draw separating lines (impossible!)
ax.plot([0.5, 0.5], [-0.5, 1.5], 'b--', alpha=0.5, label='Possible lines (all fail)')
ax.plot([-0.5, 1.5], [0.5, 0.5], 'b--', alpha=0.5)
ax.plot([-0.5, 1.5], [1.5, -0.5], 'b--', alpha=0.5)

ax.set_xlim(-0.5, 1.5)
ax.set_ylim(-0.5, 1.5)
ax.set_xlabel('Input 1')
ax.set_ylabel('Input 2')
ax.set_title('XOR Problem: Not Linearly Separable!')
ax.legend()
ax.grid(True, alpha=0.3)

plt.show()

print("\n❌ No single line can separate the green points from the red points!")
print("   This is called 'not linearly separable'.")
print("\n💡 Solution: Stack multiple perceptrons → Neural Network!")

## 6. NLP Application: Simple Word Classification

In [ ]:
# Simple feature extraction for words
def word_features(word):
    """Extract simple features from a word."""
    return np.array([
        len(word),                          # Length
        sum(1 for c in word if c in 'aeiou'),  # Vowel count
        1 if word.endswith('ing') else 0,   # Ends with 'ing'
        1 if word.endswith('ed') else 0,    # Ends with 'ed'
    ])

# Simple dataset: verbs vs nouns (simplified)
verbs = ['running', 'jumped', 'walking', 'played', 'singing', 'worked']
nouns = ['cat', 'dog', 'house', 'tree', 'book', 'car']

X_words = np.array([word_features(w) for w in verbs + nouns])
y_words = np.array([1]*len(verbs) + [0]*len(nouns))  # 1 = verb, 0 = noun

print("Word features:")
print("[length, vowels, ends_ing, ends_ed]")
for word, features, label in zip(verbs+nouns, X_words, y_words):
    print(f"  {word:10} → {features} ({'verb' if label else 'noun'})")

In [ ]:
# Normalize features
X_normalized = (X_words - X_words.mean(axis=0)) / (X_words.std(axis=0) + 1e-8)

# Train perceptron
word_perceptron = Perceptron(4, learning_rate=0.1)
history = word_perceptron.train(X_normalized, y_words, epochs=100)

print(f"\nFinal accuracy: {history[-1]:.2%}")
print(f"Weights: {word_perceptron.weights.round(2)}")
print("         [length, vowels, ends_ing, ends_ed]")

# Test on new words
test_words = ['swimming', 'cat', 'talked', 'bird']
print("\nTest predictions:")
for word in test_words:
    features = word_features(word)
    features_norm = (features - X_words.mean(axis=0)) / (X_words.std(axis=0) + 1e-8)
    pred = word_perceptron.predict(features_norm)
    print(f"  {word:10} → {'verb' if pred else 'noun'}")

## 📝 Check Your Understanding

1. What are the learnable parameters in a perceptron?
2. Why does the perceptron learning rule work?
3. What is a "linearly separable" problem?
4. Why can't a single perceptron learn XOR?
5. What's the solution to learn XOR?

In [ ]:
# --- Exercise 1: AND Gate by Hand ---
# Compute the perceptron output for AND gate with these weights.
# weights = [0.5, 0.5], bias = -0.7
# Input: [1, 1] → z = 0.5*1 + 0.5*1 + (-0.7) = 0.3 → step(0.3) = 1

ex_weights = np.array([0.5, 0.5])
ex_bias = -0.7
ex_input = np.array([1, 1])

# YOUR CODE HERE:
z = None       # Compute weighted sum + bias
output = None  # Apply step function: 1 if z >= 0, else 0

# --- Check ---
assert z is not None and output is not None, "Compute z and output!"
assert abs(z - 0.3) < 0.001, f"z = 0.5×1 + 0.5×1 - 0.7 = 0.3, got {z}"
assert output == 1, f"step(0.3) = 1 (since 0.3 ≥ 0), got {output}"
print("Exercise 1 passed! ✓")

# --- Exercise 2: XOR is Impossible ---
# Try ANY weights and bias for a single perceptron on XOR.
# Show that at least one output will be wrong.
X_xor = np.array([[0,0],[0,1],[1,0],[1,1]])
y_xor = np.array([0, 1, 1, 0])

# YOUR CODE HERE (try any values you like!):
try_weights = np.array([1.0, 1.0])  # Change these!
try_bias = -0.5                      # Change this!

# --- Check ---
outputs = step_function(X_xor @ try_weights + try_bias)
n_correct = np.sum(outputs == y_xor)
assert n_correct < 4, "If you got 4/4, you broke math! (XOR is not linearly separable)"
print(f"Exercise 2 passed! ✓  (Got {n_correct}/4 correct — XOR can't be solved with a single line!)")

# --- Quick Check: Linear Separability ---
# A problem is linearly separable when...
# a) The data points form a line
# b) You can draw one straight line (or hyperplane) to perfectly separate the classes
# c) The features are linearly correlated
# d) The model uses linear activation

your_answer = None  # Put 'a', 'b', 'c', or 'd'

# --- Check ---
assert your_answer is not None, "Pick an answer!"
assert your_answer == 'b', "Think: AND and OR can be separated by a line, XOR cannot!"
print("Exercise 3 passed! ✓")

print("\n🎉 All exercises passed!")

## 🎯 Summary

You learned:
- **Perceptron** is a single artificial neuron
- It computes: weighted sum → activation function → output
- **Can learn**: linearly separable problems (AND, OR)
- **Cannot learn**: non-linear problems (XOR)
- **Solution**: Stack multiple neurons → Multi-Layer Networks!

**Next up**: Building multi-layer networks to solve XOR! →